In [0]:
%sql

-- Show the most recent pipeline task executions

SELECT
    run_id,
    batch_id,
    layer_name,
    status,
    input_row_count,
    output_row_count,
    start_timestamp,
    end_timestamp,
    error_message
FROM online_retail_aws.control.pipeline_runs
ORDER BY
    COALESCE(end_timestamp, start_timestamp) DESC,
    layer_name
LIMIT 100;

run_id,batch_id,layer_name,status,input_row_count,output_row_count,start_timestamp,end_timestamp,error_message
demo-2010-02,2010-02,validation,SUCCESS,29388,2577,2026-09-17T07:17:46.472Z,2026-09-17T07:18:11.564Z,null
demo-2010-02,2010-02,gold,SUCCESS,29058,2577,2026-09-17T07:15:21.108Z,2026-09-17T07:16:17.250Z,null
demo-2010-02,2010-02,silver,SUCCESS,29388,29058,2026-09-17T07:12:56.789Z,2026-09-17T07:14:03.288Z,null
demo-2010-02,2010-02,bronze,SUCCESS,29388,29388,2026-09-17T07:10:52.245Z,2026-09-17T07:11:52.683Z,null
validation-cleanup-test-001,2010-02,validation,SUCCESS,29388,2577,2026-09-16T16:29:33.182Z,2026-09-16T16:30:25.856Z,null
830934714877839,2010-03,validation,SUCCESS,41511,2888,2026-09-16T06:27:46.262Z,2026-09-16T06:28:11.495Z,null
830934714877839,2010-03,gold,SUCCESS,40985,2888,2026-09-16T06:26:36.469Z,2026-09-16T06:27:30.623Z,null
830934714877839,2010-03,silver,SUCCESS,41511,40985,2026-09-16T06:25:19.508Z,2026-09-16T06:26:18.590Z,null
830934714877839,2010-03,bronze,SUCCESS,41511,41511,2026-09-16T06:23:49.712Z,2026-09-16T06:24:58.492Z,null
188028901134264,2010-03,bronze,FAILED,null,null,2026-09-16T06:11:14.060Z,2026-09-16T06:12:21.229Z,RunExecutionError


In [0]:
%sql

-- Show tasks that did not complete successfully

SELECT
    run_id,
    batch_id,
    layer_name,
    status,
    start_timestamp,
    end_timestamp,
    error_message
FROM online_retail_aws.control.pipeline_runs
WHERE status <> 'SUCCESS'
ORDER BY
    COALESCE(end_timestamp, start_timestamp) DESC,
    layer_name;

run_id,batch_id,layer_name,status,start_timestamp,end_timestamp,error_message
188028901134264,2010-03,bronze,FAILED,2026-09-16T06:11:14.060Z,2026-09-16T06:12:21.229Z,RunExecutionError
188028901134264,2010-03,gold,UPSTREAM_FAILED,null,2026-09-16T06:12:21.229Z,Skipped
188028901134264,2010-03,silver,UPSTREAM_FAILED,null,2026-09-16T06:12:21.229Z,Skipped
188028901134264,2010-03,validation,UPSTREAM_FAILED,null,2026-09-16T06:12:21.229Z,Skipped


In [0]:
%sql

-- Summarize the recorded task results for each pipeline run

SELECT
    run_id,
    batch_id,
    COUNT(*) AS recorded_task_count,
    SUM(
        CASE
            WHEN status = 'SUCCESS' THEN 1
            ELSE 0
        END
    ) AS successful_task_count,
    SUM(
        CASE
            WHEN status <> 'SUCCESS' THEN 1
            ELSE 0
        END
    ) AS unsuccessful_task_count,
    MIN(start_timestamp) AS pipeline_start_timestamp,
    MAX(end_timestamp) AS pipeline_end_timestamp
FROM online_retail_aws.control.pipeline_runs
GROUP BY
    run_id,
    batch_id
ORDER BY pipeline_end_timestamp DESC;

run_id,batch_id,recorded_task_count,successful_task_count,unsuccessful_task_count,pipeline_start_timestamp,pipeline_end_timestamp
demo-2010-02,2010-02,4,4,0,2026-09-17T07:10:52.245Z,2026-09-17T07:18:11.564Z
validation-cleanup-test-001,2010-02,1,1,0,2026-09-16T16:29:33.182Z,2026-09-16T16:30:25.856Z
830934714877839,2010-03,4,4,0,2026-09-16T06:23:49.712Z,2026-09-16T06:28:11.495Z
188028901134264,2010-03,4,0,4,2026-09-16T06:11:14.060Z,2026-09-16T06:12:21.229Z
331135983682346,2010-02,4,4,0,2026-09-16T06:06:19.771Z,2026-09-16T06:09:56.017Z
67118216817424,2010-01,4,4,0,2026-09-16T06:01:59.973Z,2026-09-16T06:05:36.995Z
683068586050848,2009-12,4,4,0,2026-09-16T05:57:11.708Z,2026-09-16T06:01:09.193Z
972836784151481,2010-01,4,4,0,2026-09-16T05:35:49.950Z,2026-09-16T05:39:48.646Z
gold-cleanup-test-002,2010-02,1,1,0,2026-09-16T05:23:11.997Z,2026-09-16T05:23:56.379Z
gold-cleanup-test-001,2010-02,1,1,0,2026-09-16T05:18:33.688Z,2026-09-16T05:19:54.866Z
